In [3]:
!pip install boto3 --break-system-packages


Defaulting to user installation because normal site-packages is not writeable


In [6]:
%pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 145.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [boto3]32m3/4 [boto3]re]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /home/ubuntu/jupyter_env/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
%pip install s3fs

  Using cached s3fs-2026.6.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached aiobotocore-3.8.0-py3-none-any.whl.metadata (29 kB)
  Using cached aioitertools-0.13.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached botocore-1.43.46-py3-none-any.whl.metadata (5.6 kB)
  Using cached wrapt-2.2.2-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (7.4 kB)
Using cached s3fs-2026.6.0-py3-none-any.whl (32 kB)
Using cached aiobotocore-3.8.0-py3-none-any.whl (91 kB)
Using cached aioitertools-0.13.0-py3-none-any.whl (24 kB)
Using cached botocore-1.43.46-py3-none-any.whl (15.4 MB)
Using cached wrapt-2.2.2-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (167 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: botocorem━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [fsspec]
    Found existing installation:

In [25]:
!pip install s3fs --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


In [32]:
import boto3
from botocore.exceptions import ClientError

# =====================================================================
# 1. CONFIGURATION
# =====================================================================
ENDPOINT_URL = "https://ef8eef61229ee8854b4237f6949e50d8.r2.cloudflarestorage.com/truestates-re-analytics"
ACCESS_KEY_ID = "c198c85bd01da0931eae24009fb2100b"
SECRET_ACCESS_KEY = "826187ffaee4742816f65ca4ebe149902db75ac52dbb81606bb34fe8bae4a57c"

BUCKET_NAME = "dubai"
LOCAL_FILE_TO_UPLOAD = r"/home/ubuntu/Dubai/data/raw/project_scorecard.xlsx"
R2_OBJECT_NAME = "folder1/folder2/uploaded_truintel.py"
LOCAL_FILE_TO_DOWNLOAD = r"/home/ubuntu/Dubai/data/raw/project_scorecard.xlsx"

# =====================================================================
# 2. INITIALIZE CLIENT
# =====================================================================
    
# Initialize the client with the current region in the loop
s3_client = boto3.client(
    service_name="s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY_ID,
    aws_secret_access_key=SECRET_ACCESS_KEY,
    region_name="auto"
)


# =====================================================================
# 3. CREATE A BUCKET
# =====================================================================
def create_r2_bucket(bucket_name):
    try:
        print(f"Creating bucket: {bucket_name}...")
        s3_client.create_bucket(Bucket=bucket_name)
        print(f"Success: Bucket '{bucket_name}' created successfully.")
    except ClientError as e:
        print(f"Error creating bucket: {e}")


# =====================================================================
# 4. ADD / UPLOAD A FILE
# =====================================================================
def upload_file_to_r2(local_path, bucket_name, r2_key):
    try:
        print(f"Uploading '{local_path}' to '{bucket_name}' as '{r2_key}'...")
        s3_client.upload_file(local_path, bucket_name, r2_key)
        print("Success: File uploaded successfully.")
    except FileNotFoundError:
        print(f"Error: The local file '{local_path}' was not found.")
    except ClientError as e:
        print(f"Error uploading file: {e}")

# =====================================================================
# 5. PULL / DOWNLOAD A FILE
# =====================================================================
def download_file_from_r2(bucket_name, r2_key, local_path):
    try:
        print(f"Downloading '{r2_key}' from '{bucket_name}' to '{local_path}'...")
        s3_client.download_file(bucket_name, r2_key, local_path)
        print("Success: File downloaded successfully.")
    except ClientError as e:
        print(f"Error downloading file: {e}")





In [13]:
# 1. Create the bucket
create_r2_bucket(BUCKET_NAME)

# 2. Upload the file
upload_file_to_r2(LOCAL_FILE_TO_UPLOAD, BUCKET_NAME, R2_OBJECT_NAME)

# 3. Pull the file back down
download_file_from_r2(BUCKET_NAME, R2_OBJECT_NAME, LOCAL_FILE_TO_DOWNLOAD)

Creating bucket: dubai...
Success: Bucket 'dubai' created successfully.
Uploading '/home/ubuntu/Dubai/data/raw/project_scorecard.xlsx' to 'dubai' as 'folder1/folder2/uploaded_truintel.py'...
Success: File uploaded successfully.
Success: File downloaded successfully.


#https://truestates-re-ai.blobs.truestates.ai/test_bucket1/folder1/folder2/uploaded_truintel.py - this is the public link to the file that was uploaded to R2. You can use this link to access the file directly from a web browser or any HTTP client.This can also be used to verify that the file was uploaded successfully to the R2 bucket. This can also be used as a url to download files as well.


In [15]:
import os

def upload_directory_to_r2(local_dir, bucket_name, r2_prefix):
    """
    Uploads an entire local directory to an R2 bucket while preserving the folder structure.
    """
    print(f"Scanning local directory: {local_dir}")
    
    for root, dirs, files in os.walk(local_dir):
        for file in files:
            # 1. Get the exact local file path
            local_path = os.path.join(root, file)
            
            # 2. Calculate the relative path (to keep subfolders intact)
            relative_path = os.path.relpath(local_path, local_dir)
            
            # 3. Create the target path inside the R2 bucket
            r2_key = f"{r2_prefix}/{relative_path}".replace("\\", "/")
            
            print(f"Uploading: {file} -> s3://{bucket_name}/{r2_key}")
            
            # 4. Upload the file
            s3_client.upload_file(local_path, bucket_name, r2_key)
            
    print("✅ All raw files uploaded successfully!")

# --- Configuration matching your config.yaml ---
LOCAL_RAW_DIR = r"/home/ubuntu/Dubai/data/raw"
R2_TARGET_PREFIX = "data/raw"
BUCKET = "dubai"

# Run the upload
upload_directory_to_r2(LOCAL_RAW_DIR, BUCKET, R2_TARGET_PREFIX)

Scanning local directory: /home/ubuntu/Dubai/data/raw
Uploading: projects.parquet -> s3://dubai/data/raw/projects.parquet
Uploading: units.parquet -> s3://dubai/data/raw/units.parquet
Uploading: developers.parquet -> s3://dubai/data/raw/developers.parquet
Uploading: buildings.parquet -> s3://dubai/data/raw/buildings.parquet
Uploading: Dubai_RE_Classification_DLD.xlsx -> s3://dubai/data/raw/Dubai_RE_Classification_DLD.xlsx
Uploading: developer_scorecard.xlsx -> s3://dubai/data/raw/developer_scorecard.xlsx
Uploading: transactions.parquet -> s3://dubai/data/raw/transactions.parquet
Uploading: project_scorecard.xlsx -> s3://dubai/data/raw/project_scorecard.xlsx
✅ All raw files uploaded successfully!


In [16]:
import random
import os

# 1. Fetch the list of all files uploaded to the 'dubai' bucket
response = s3_client.list_objects_v2(Bucket="dubai", Prefix="data/raw/")

if "Contents" in response and len(response["Contents"]) > 0:
    # 2. Pick a random file key from the bucket
    file_keys = [obj["Key"] for obj in response["Contents"]]
    random_key = random.choice(file_keys)
    
    # 3. Define a local destination path (e.g. into /tmp)
    file_name = os.path.basename(random_key)
    local_dest = f"/tmp/{file_name}"
    
    print(f"🎲 Selected random file from R2: {random_key}")
    print(f"📥 Downloading to: {local_dest} ...")
    
    # 4. Download file from Cloudflare R2
    s3_client.download_file("dubai", random_key, local_dest)
    
    # 5. Verify file exists locally and print size
    file_size_mb = os.path.getsize(local_dest) / (1024 * 1024)
    print(f"✅ Success! File downloaded successfully ({file_size_mb:.2f} MB).")
else:
    print("⚠️ No objects found in bucket 'dubai' under prefix 'data/raw/'")

⚠️ No objects found in bucket 'dubai' under prefix 'data/raw/'


In [18]:
import os

+
# Change to any local path you prefer

try:
    print(f"📥 Downloading '{R2_FILE_KEY}' from bucket '{BUCKET_NAME}'...")
    s3_client.download_file(BUCKET_NAME, R2_FILE_KEY, LOCAL_DESTINATION)
    
    file_size_mb = os.path.getsize(LOCAL_DESTINATION) / (1024 * 1024)
    print(f"✅ Download complete! File saved to: {LOCAL_DESTINATION} ({file_size_mb:.2f} MB)")
    
except Exception as e:
    print(f"❌ Failed to download file: {e}")

📥 Downloading 'data/raw/project_scorecard.xlsx' from bucket 'dubai'...
✅ Download complete! File saved to: /tmp/project_scorecard.xlsx (0.18 MB)


In [35]:
import s3fs

# Initialize the S3 file system WITH your R2 credentials
fs = s3fs.S3FileSystem(
    key="c198c85bd01da0931eae24009fb2100b",
    secret="826187ffaee4742816f65ca4ebe149902db75ac52dbb81606bb34fe8bae4a57c",
    client_kwargs={
        'endpoint_url': 'https://ef8eef61229ee8854b4237f6949e50d8.r2.cloudflarestorage.com/truestates-re-analytics'
    }
)

# Use your exact absolute path here:
local_utils_folder = "/home/ubuntu/Dubai/data/utils/"
r2_destination = "s3://dubai/data/utils/"

# Push the entire folder to R2
fs.put(local_utils_folder, r2_destination, recursive=True)

print("✅ Utils folder successfully uploaded to R2!")

✅ Utils folder successfully uploaded to R2!


In [37]:
import s3fs
import json

# 1. Initialize s3fs with the strict R2 auth settings
fs = s3fs.S3FileSystem(
    key="c198c85bd01da0931eae24009fb2100b",
    secret="826187ffaee4742816f65ca4ebe149902db75ac52dbb81606bb34fe8bae4a57c",
    client_kwargs={
        'endpoint_url': 'https://ef8eef61229ee8854b4237f6949e50d8.r2.cloudflarestorage.com/truestates-re-analytics',
        'region_name': 'auto'
    },
    config_kwargs={'signature_version': 's3v4'}
)

# 2. Define the exact files we expect to be in the bucket
files_to_test = [
    "s3://dubai/data/utils/entity_tiers.json",
    "s3://dubai/data/utils/event_scale_bins.json",
    "s3://dubai/data/utils/geo_relevance.json",
    "s3://dubai/data/utils/event_to_channel.json",
    "s3://dubai/data/utils/area_sensitivity.json"
]

print("🔍 Testing R2 JSON File Reads...\n")

# 3. Loop through and try to read each one
for file_path in files_to_test:
    try:
        if fs.exists(file_path):
            with fs.open(file_path, 'rb') as f:
                data = json.load(f)
                print(f"✅ SUCCESS: {file_path}")
                # Print the first few keys to prove the data is actually there
                print(f"   ↳ Found {len(data)} items. Top keys: {list(data.keys())[:5]}\n")
        else:
            print(f"❌ NOT FOUND: {file_path} (File is missing from bucket)\n")
    except Exception as e:
        print(f"🚫 ERROR READING {file_path}:\n   ↳ {e}\n")

🔍 Testing R2 JSON File Reads...

✅ SUCCESS: s3://dubai/data/utils/entity_tiers.json
   ↳ Found 5 items. Top keys: ['T1', 'T2', 'T3', 'T4', 'Default']

✅ SUCCESS: s3://dubai/data/utils/event_scale_bins.json
   ↳ Found 5 items. Top keys: ['mega', 'large', 'medium', 'small', 'Default']

✅ SUCCESS: s3://dubai/data/utils/geo_relevance.json
   ↳ Found 5 items. Top keys: ['uae_specific', 'middle_east', 'global_macro', 'foreign_domestic', 'Default']

✅ SUCCESS: s3://dubai/data/utils/event_to_channel.json
   ↳ Found 28 items. Top keys: ['TOURISM_MEGA_PROJECT', 'TOURISM_FOOTFALL_SURGE', 'AIR_CAPACITY_EXPANSION', 'GLOBAL_TOURISM_SHOCK', 'MAJOR_INFRA_PROJECT']

✅ SUCCESS: s3://dubai/data/utils/area_sensitivity.json
   ↳ Found 30 items. Top keys: ['AL_BARSHAA_SOUTH_FIFTH', 'AL_BARSHAA_SOUTH_FOURTH', 'AL_BARSHAA_SOUTH_SECOND', 'AL_BARSHAA_SOUTH_THIRD', 'AL_HEBIAH_FIRST']

